# Experiment 2: Basic Tensor Operations

**Course:** AML ZC417 – Introduction to Deep Learning
**Module Reference:** Module 3 — Data Representation and Deep Learning Frameworks
**Duration:** 2 hours

---

## Aim
To perform and understand core tensor operations — broadcasting, indexing/slicing, concatenation/stacking, axis-wise reduction, and automatic differentiation — using TensorFlow and PyTorch, building on the tensor basics introduced in Experiment 1.

## Learning Outcomes
By the end of this experiment, you will be able to:
1. Apply broadcasting rules to operate on tensors of different shapes.
2. Perform advanced indexing and slicing on multi-dimensional tensors.
3. Concatenate and stack tensors along different axes.
4. Compute axis-wise reductions (sum, mean, max, argmax) correctly.
5. Distinguish in-place from out-of-place tensor operations.
6. Explain and demonstrate automatic differentiation (autograd) on a simple expression.


## Part 0 — Setup

Run the cell below to import the libraries needed for this experiment.


In [72]:
import numpy as np
import tensorflow as tf
import torch

print("TensorFlow version:", tf.__version__)
print("PyTorch version:", torch.__version__)


TensorFlow version: 2.21.0
PyTorch version: 2.13.0+cpu


---
## Part 1 — Broadcasting

**Broadcasting** lets operations work on tensors of different shapes without explicitly copying data, by "stretching" the smaller tensor's dimensions to match the larger one — provided the shapes are *compatible*.

**Broadcasting rule:** Two dimensions are compatible when they are equal, or one of them is 1. Dimensions are compared from the **rightmost** dimension inward.


### 1.1 Broadcasting in TensorFlow

In [73]:
# A (3,3) matrix and a (3,) vector -- vector is broadcast across each row
matrix = tf.constant([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=tf.float32)
vector = tf.constant([10, 20, 30], dtype=tf.float32)

result = matrix + vector
print("Matrix:\n", matrix.numpy())
print("Vector:", vector.numpy())
print("Matrix + Vector (broadcast):\n", result.numpy())


Matrix:
 [[1. 2. 3.]
 [4. 5. 6.]
 [7. 8. 9.]]
Vector: [10. 20. 30.]
Matrix + Vector (broadcast):
 [[11. 22. 33.]
 [14. 25. 36.]
 [17. 28. 39.]]


In [74]:
# A (3,1) column vector broadcast against a (1,3) row vector -> (3,3) result
col = tf.constant([[1], [2], [3]], dtype=tf.float32)   # shape (3,1)
row = tf.constant([10, 20, 30], dtype=tf.float32)    # shape (1,3)

print("col shape:", col.shape, "| row shape:", row.shape)
print("col + row =\n", (col + row).numpy())


col shape: (3, 1) | row shape: (3,)
col + row =
 [[11. 21. 31.]
 [12. 22. 32.]
 [13. 23. 33.]]


### 1.2 Broadcasting in PyTorch

In [75]:
matrix_pt = torch.tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=torch.float32)
vector_pt = torch.tensor([10, 20, 30], dtype=torch.float32)

result_pt = matrix_pt + vector_pt
print("Matrix:\n", matrix_pt)
print("Vector:", vector_pt)
print("Matrix + Vector (broadcast):\n", result_pt)


Matrix:
 tensor([[1., 2., 3.],
        [4., 5., 6.],
        [7., 8., 9.]])
Vector: tensor([10., 20., 30.])
Matrix + Vector (broadcast):
 tensor([[11., 22., 33.],
        [14., 25., 36.],
        [17., 28., 39.]])


In [76]:
col_pt = torch.tensor([[1], [2], [3]], dtype=torch.float32)
row_pt = torch.tensor([[10, 20, 30]], dtype=torch.float32)

print("col shape:", col_pt.shape, "| row shape:", row_pt.shape)
print("col + row =\n", col_pt + row_pt)


col shape: torch.Size([3, 1]) | row shape: torch.Size([1, 3])
col + row =
 tensor([[11., 21., 31.],
        [12., 22., 32.],
        [13., 23., 33.]])


**Try it yourself:** Predict the output shape of a `(4, 1, 3)` tensor broadcast against a `(1, 5, 3)` tensor, then verify using `tf.broadcast_static_shape` or by just adding the two tensors together.


In [77]:
a = tf.ones((4, 1, 3))
b = tf.ones((1, 5, 3))
c = a + b
print(c)
print("Result shape:", c.shape)  # Expected: (4, 5, 3)


tf.Tensor(
[[[2. 2. 2.]
  [2. 2. 2.]
  [2. 2. 2.]
  [2. 2. 2.]
  [2. 2. 2.]]

 [[2. 2. 2.]
  [2. 2. 2.]
  [2. 2. 2.]
  [2. 2. 2.]
  [2. 2. 2.]]

 [[2. 2. 2.]
  [2. 2. 2.]
  [2. 2. 2.]
  [2. 2. 2.]
  [2. 2. 2.]]

 [[2. 2. 2.]
  [2. 2. 2.]
  [2. 2. 2.]
  [2. 2. 2.]
  [2. 2. 2.]]], shape=(4, 5, 3), dtype=float32)
Result shape: (4, 5, 3)


---
## Part 2 — Advanced Indexing and Slicing

### 2.1 TensorFlow


In [78]:
t = tf.constant(np.arange(24).reshape(4, 6))
print("Tensor:\n", t.numpy())

print("\nRow 2:", t[2].numpy())
print("Column 3:", t[:, 3].numpy())
print("Sub-matrix (rows 1-2, cols 2-4):\n", t[1:3, 2:5].numpy())
print("Every other row:\n", t[::2].numpy())
print("Reversed rows:\n", t[::-1].numpy())


Tensor:
 [[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]
 [12 13 14 15 16 17]
 [18 19 20 21 22 23]]

Row 2: [12 13 14 15 16 17]
Column 3: [ 3  9 15 21]
Sub-matrix (rows 1-2, cols 2-4):
 [[ 8  9 10]
 [14 15 16]]
Every other row:
 [[ 0  1  2  3  4  5]
 [12 13 14 15 16 17]]
Reversed rows:
 [[18 19 20 21 22 23]
 [12 13 14 15 16 17]
 [ 6  7  8  9 10 11]
 [ 0  1  2  3  4  5]]


In [79]:
# Boolean masking in TensorFlow
t2 = tf.constant([1, -2, 3, -4, 5, -6])
mask = t2 > 0
print("Mask:", mask.numpy())
print("Positive values only:", tf.boolean_mask(t2, mask).numpy())


Mask: [ True False  True False  True False]
Positive values only: [1 3 5]


### 2.2 PyTorch

In [80]:
t_pt = torch.arange(24).reshape(4, 6)
print("Tensor:\n", t_pt)

print("\nRow 2:", t_pt[2])
print("Column 3:", t_pt[:, 3])
print("Sub-matrix (rows 1-2, cols 2-4):\n", t_pt[1:3, 2:5])
print("Every other row:\n", t_pt[::2])
print("Reversed rows:\n", t_pt.flip(0))


Tensor:
 tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11],
        [12, 13, 14, 15, 16, 17],
        [18, 19, 20, 21, 22, 23]])

Row 2: tensor([12, 13, 14, 15, 16, 17])
Column 3: tensor([ 3,  9, 15, 21])
Sub-matrix (rows 1-2, cols 2-4):
 tensor([[ 8,  9, 10],
        [14, 15, 16]])
Every other row:
 tensor([[ 0,  1,  2,  3,  4,  5],
        [12, 13, 14, 15, 16, 17]])
Reversed rows:
 tensor([[18, 19, 20, 21, 22, 23],
        [12, 13, 14, 15, 16, 17],
        [ 6,  7,  8,  9, 10, 11],
        [ 0,  1,  2,  3,  4,  5]])


In [81]:
# Boolean masking in PyTorch
t2_pt = torch.tensor([1, -2, 3, -4, 5, -6])
mask_pt = t2_pt > 0
print("Mask:", mask_pt)
print("Positive values only:", t2_pt[mask_pt])


Mask: tensor([ True, False,  True, False,  True, False])
Positive values only: tensor([1, 3, 5])


> **Note:** PyTorch does not support negative-step slicing (`::-1`) directly on tensors the way NumPy/TensorFlow do — `.flip(dim)` is the PyTorch equivalent for reversing along an axis.


---
## Part 3 — Concatenation and Stacking

- **Concatenate**: joins tensors along an *existing* axis (output rank stays the same).
- **Stack**: joins tensors along a *new* axis (output rank increases by 1).

### 3.1 TensorFlow


In [82]:
a = tf.constant([[1, 2], [3, 4]])
b = tf.constant([[5, 6], [7, 8]])

concat_axis0 = tf.concat([a, b], axis=0)
concat_axis1 = tf.concat([a, b], axis=1)
stacked = tf.stack([a, b], axis=0)

print("Concat along axis 0 (rows):\n", concat_axis0.numpy())
print("Shape:", concat_axis0.shape)
print("\nConcat along axis 1 (columns):\n", concat_axis1.numpy())
print("Shape:", concat_axis1.shape)
print("\nStacked (new axis 0):\n", stacked.numpy())
print("Shape:", stacked.shape)


Concat along axis 0 (rows):
 [[1 2]
 [3 4]
 [5 6]
 [7 8]]
Shape: (4, 2)

Concat along axis 1 (columns):
 [[1 2 5 6]
 [3 4 7 8]]
Shape: (2, 4)

Stacked (new axis 0):
 [[[1 2]
  [3 4]]

 [[5 6]
  [7 8]]]
Shape: (2, 2, 2)


### 3.2 PyTorch

In [83]:
a_pt = torch.tensor([[1, 2], [3, 4]])
b_pt = torch.tensor([[5, 6], [7, 8]])

concat_axis0_pt = torch.cat([a_pt, b_pt], dim=0)
concat_axis1_pt = torch.cat([a_pt, b_pt], dim=1)
stacked_pt = torch.stack([a_pt, b_pt], dim=0)

print("Concat along dim 0 (rows):\n", concat_axis0_pt)
print("Shape:", concat_axis0_pt.shape)
print("\nConcat along dim 1 (columns):\n", concat_axis1_pt)
print("Shape:", concat_axis1_pt.shape)
print("\nStacked (new dim 0):\n", stacked_pt)
print("Shape:", stacked_pt.shape)


Concat along dim 0 (rows):
 tensor([[1, 2],
        [3, 4],
        [5, 6],
        [7, 8]])
Shape: torch.Size([4, 2])

Concat along dim 1 (columns):
 tensor([[1, 2, 5, 6],
        [3, 4, 7, 8]])
Shape: torch.Size([2, 4])

Stacked (new dim 0):
 tensor([[[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]]])
Shape: torch.Size([2, 2, 2])


---
## Part 4 — Axis-Wise Reductions

Reduction operations (sum, mean, max, argmax, …) collapse a tensor along one or more axes. The `axis` (TensorFlow) / `dim` (PyTorch) parameter controls **which** dimension gets collapsed — this is one of the most common sources of bugs for beginners, so study the examples carefully.

### 4.1 TensorFlow


In [84]:
m = tf.constant([[1, 2, 3], [4, 5, 6]], dtype=tf.float32)
print("Matrix:\n", m.numpy())

print("\nSum of all elements:", tf.reduce_sum(m).numpy())
print("Sum along axis=0 (collapse rows, per column):", tf.reduce_sum(m, axis=0).numpy())
print("Sum along axis=1 (collapse columns, per row):", tf.reduce_sum(m, axis=1).numpy())

print("\nMean along axis=0:", tf.reduce_mean(m, axis=0).numpy())
print("Max along axis=1:", tf.reduce_max(m, axis=1).numpy())
print("Index of max along axis=1 (argmax):", tf.argmax(m, axis=1).numpy())


Matrix:
 [[1. 2. 3.]
 [4. 5. 6.]]

Sum of all elements: 21.0
Sum along axis=0 (collapse rows, per column): [5. 7. 9.]
Sum along axis=1 (collapse columns, per row): [ 6. 15.]

Mean along axis=0: [2.5 3.5 4.5]
Max along axis=1: [3. 6.]
Index of max along axis=1 (argmax): [2 2]


### 4.2 PyTorch

In [85]:
m_pt = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)
print("Matrix:\n", m_pt)

print("\nSum of all elements:", m_pt.sum().item())
print("Sum along dim=0 (collapse rows, per column):", m_pt.sum(dim=0))
print("Sum along dim=1 (collapse columns, per row):", m_pt.sum(dim=1))

print("\nMean along dim=0:", m_pt.mean(dim=0))
print("Max along dim=1:", m_pt.max(dim=1).values)
print("Index of max along dim=1 (argmax):", m_pt.argmax(dim=1))


Matrix:
 tensor([[1., 2., 3.],
        [4., 5., 6.]])

Sum of all elements: 21.0
Sum along dim=0 (collapse rows, per column): tensor([5., 7., 9.])
Sum along dim=1 (collapse columns, per row): tensor([ 6., 15.])

Mean along dim=0: tensor([2.5000, 3.5000, 4.5000])
Max along dim=1: tensor([3., 6.])
Index of max along dim=1 (argmax): tensor([2, 2])


---
## Part 5 — In-Place vs. Out-of-Place Operations

- **Out-of-place**: returns a *new* tensor; the original is unchanged.
- **In-place**: modifies the tensor's data directly (no new memory allocated). PyTorch marks in-place ops with a trailing underscore, e.g. `add_()`.

In-place operations save memory but can cause subtle bugs — especially when a tensor is needed later for gradient computation. Use them carefully.

### 5.1 PyTorch (TensorFlow constants are immutable, so this concept is most relevant to `tf.Variable` and PyTorch tensors)


In [86]:
x = torch.tensor([1.0, 2.0, 3.0])
y = x.add(10)          # out-of-place: x is unchanged, y is new
print("x after out-of-place add:", x)
print("y (new tensor):", y)

x.add_(10)             # in-place: x itself is modified
print("x after in-place add_:", x)


x after out-of-place add: tensor([1., 2., 3.])
y (new tensor): tensor([11., 12., 13.])
x after in-place add_: tensor([11., 12., 13.])


### 5.2 Mutability in TensorFlow — `tf.constant` vs `tf.Variable`

In [87]:
const = tf.constant([1.0, 2.0, 3.0])
try:
    const[0].assign(99)   # this will fail -- constants cannot be modified
except AttributeError as e:
    print("Error (expected):", e)

var = tf.Variable([1.0, 2.0, 3.0])
var[0].assign(99.0)        # tf.Variable supports in-place-style modification
print("Variable after assign:", var.numpy())


Error (expected): 'tensorflow.python.framework.ops.EagerTensor' object has no attribute 'assign'
Variable after assign: [99.  2.  3.]


---
## Part 6 — Automatic Differentiation (Autograd)

Every deep learning framework can automatically compute gradients — this is what makes training neural networks via backpropagation possible (covered in detail in Module 4). Here we compute the gradient of a simple scalar function by hand-verification.

Let **y = x² + 3x + 5**. Then **dy/dx = 2x + 3**. At x = 4, dy/dx = 2(4) + 3 = **11**.

### 6.1 Autograd in TensorFlow (`tf.GradientTape`)


In [88]:
x = tf.Variable(4.0)

with tf.GradientTape() as tape:
    y = x**2 + 3*x + 5

dy_dx = tape.gradient(y, x)
print("y =", y.numpy())
print("dy/dx at x=4:", dy_dx.numpy(), "(expected: 11)")


y = 33.0
dy/dx at x=4: 11.0 (expected: 11)


### 6.2 Autograd in PyTorch (`.backward()`)

In [89]:
x_pt = torch.tensor(4.0, requires_grad=True)

y_pt = x_pt**2 + 3*x_pt + 5
y_pt.backward()

print("y =", y_pt.item())
print("dy/dx at x=4:", x_pt.grad.item(), "(expected: 11)")


y = 33.0
dy/dx at x=4: 11.0 (expected: 11)


**Key idea:** TensorFlow requires a `tf.GradientTape()` context to *record* operations for differentiation, and you call `tape.gradient(output, input)` afterward. PyTorch tracks operations automatically on any tensor created with `requires_grad=True`, and you call `.backward()` on the output, after which the gradient is available in `.grad`.


---
## Part 7 — Side-by-Side Comparison

| Operation | TensorFlow | PyTorch |
|---|---|---|
| Broadcasting | Automatic (NumPy-style rules) | Automatic (NumPy-style rules) |
| Boolean masking | `tf.boolean_mask(t, mask)` | `t[mask]` |
| Reverse along axis | `t[::-1]` | `t.flip(dim)` |
| Concatenate | `tf.concat([a,b], axis=0)` | `torch.cat([a,b], dim=0)` |
| Stack (new axis) | `tf.stack([a,b], axis=0)` | `torch.stack([a,b], dim=0)` |
| Sum along axis | `tf.reduce_sum(t, axis=0)` | `t.sum(dim=0)` |
| Argmax along axis | `tf.argmax(t, axis=1)` | `t.argmax(dim=1)` |
| Mutable tensor | `tf.Variable` | any tensor (in-place ops end in `_`) |
| Autograd | `tf.GradientTape()` + `tape.gradient()` | `requires_grad=True` + `.backward()` + `.grad` |

### Key takeaway
Both frameworks follow the **same NumPy-derived conventions** for broadcasting, reductions, and axis semantics — the syntax differs (`axis` vs `dim`, `tf.reduce_sum` vs `.sum()`) but the underlying mathematics is identical. Autograd is the one place where the *programming model* genuinely differs: TensorFlow's tape-based recording vs. PyTorch's always-on tracking of `requires_grad` tensors.


---
## Part 8 — In-Lab Exercise (to be completed and shown to the instructor)

Complete the following in the empty cells below, in **both** TensorFlow and PyTorch:

1. Create a `(3, 4)` tensor of integers 1–12 (in order) and a `(4,)` vector `[1, 0, 1, 0]`. Add them using broadcasting and print the result.
2. From the `(3, 4)` tensor above, extract the sub-tensor consisting of the **last two rows and last two columns**.
3. Create two `(2, 3)` tensors of your choice. Concatenate them along axis 0, then stack them along a new axis 0. Print both shapes and explain (in a markdown cell) why they differ.
4. Compute the **column-wise mean** and the **row-wise max** of the `(3, 4)` tensor from Step 1.
5. Let **f(x) = 3x³ − 2x + 1**. Using autograd, compute df/dx at x = 2 in both frameworks, and verify it matches the hand-calculated derivative **f'(x) = 9x² − 2**.


In [90]:
# TODO 1: Broadcasting exercise -- TensorFlow
# your code here


In [91]:
# TODO 1: Broadcasting exercise -- PyTorch
# your code here


In [92]:
# TODO 2: Sub-tensor extraction (both frameworks)
# your code here


In [93]:
# TODO 3: Concatenate vs stack (both frameworks)
# your code here


In [94]:
# TODO 4: Column-wise mean and row-wise max (both frameworks)
# your code here


In [95]:
# TODO 5: Autograd for f(x) = 3x^3 - 2x + 1 at x=2 -- TensorFlow
# your code here


In [96]:
# TODO 5: Autograd for f(x) = 3x^3 - 2x + 1 at x=2 -- PyTorch
# your code here
